In [ ]:
#读取音频并展示
from pathlib import Path
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent

candidate_path = root / "metadata" / "test_candidates.csv"

audio_root = (
    root / "data" / "raw"
    / "common_voice_beijing26"
    / "zh-CN-beijing"
)

def resolve_audio_path(audio_path):
    p = Path(str(audio_path))

    candidates = (
        [p] if p.is_absolute() else [
            root / p,
            audio_root / p,
            audio_root / "clips" / p
        ]
    )

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"找不到音频:{audio_path}"
    )

def show_candidate(rank):
    df = pd.read_csv(
        candidate_path,
        keep_default_na=False
    )

    selected = df.loc[
        df["selection_rank"] == rank
    ]

    if len(selected) != 1:
        raise ValueError("没有找到唯一候选样本")

    row = selected.iloc[0]

    if row["review_batch"] != "W5D4":
        raise ValueError("该样本不属于W5D4")

    path = resolve_audio_path(row["audio_path"])

    #读取音频
    audio, sr = sf.read(
        path,
        dtype="float32",
        always_2d=True
    )

    #如果有多个声道，简单取平均
    waveform = audio.mean(axis=1)

    print("序号:", rank)
    print("UID:", row["candidate_uid"])
    print("音频时长:", len(waveform) / sr)
    print("当前状态:", row["status"])
    print("原始转写:", row["source_text"])

    display(Audio(waveform, rate=sr))

show_candidate(1)

序号: 1
UID: cv26_cdee3e6a07874a70
音频时长: 6.432
当前状态: pending
原始转写: 智秀深信不疑丈夫俊彪是一个标准的好男人。


In [2]:
#标注更新函数
def update_candidate(
        rank,
        raw_text,
        status,
        uncertain_span="",
        notes="",
        review_minutes=""
):
    if status not in ["verified","needs_review"]:
        raise ValueError("无效标注状态")

    if status == "verified":
        if not str(raw_text).strip():
            raise ValueError("verified必须有确认后的文本")

        if str(uncertain_span).strip():
            raise ValueError(
                "仍存在不确定的片段，不能标记verified"
            )

    if status == "needs_review":
        if not str(uncertain_span).strip() and not str(notes).strip():
            raise ValueError("请记录需要复核的原因")

    df = pd.read_csv(
        candidate_path,
        dtype=str,
        keep_default_na=False
    )

    mask = df["selection_rank"] == str(rank)

    if mask.sum() != 1:
        raise ValueError("没有找到唯一候选样本")

    if df.loc[mask, "review_batch"].iloc[0] != "W5D4":
        raise ValueError("该样本不属于W5D4")

    df.loc[mask, "raw_text"] = raw_text
    df.loc[mask, "status"] = status
    df.loc[mask, "uncertain_span"] = uncertain_span
    df.loc[mask, "notes"] = notes
    df.loc[mask, "review_minutes"] = str(review_minutes)

    #临时文件写入成功后，再替换原CSV
    temp_path = candidate_path.with_suffix(".csv.tmp")

    df.to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig"
    )

    temp_path.replace(candidate_path)

    print(f"第{rank}条已更新为：{status}")

In [ ]:
#首先听音频
for i in range(1,11):
    show_candidate(i)

序号: 1
UID: cv26_cdee3e6a07874a70
音频时长: 6.432
当前状态: pending
原始转写: 智秀深信不疑丈夫俊彪是一个标准的好男人。


序号: 2
UID: cv26_aed66ddddae351fa
音频时长: 5.472
当前状态: pending
原始转写: 黄翅蜻属是蜻蜓科下的一个属。


序号: 3
UID: cv26_ed5ccf1debf56423
音频时长: 5.4
当前状态: pending
原始转写: 元贞年间再改为横州。


序号: 4
UID: cv26_6077d56f73cbebcb
音频时长: 9.672
当前状态: pending
原始转写: 基督教法西斯主义是基督教和法西斯主义的有争议混成词。


序号: 5
UID: cv26_325f8ba312c58e72
音频时长: 8.28
当前状态: pending
原始转写: 它的人员也常因非法闯入这一类罪行被逮捕。


序号: 6
UID: cv26_79fe04b3101be4c2
音频时长: 6.72
当前状态: pending
原始转写: 它同时还能够遏制企业在外州建厂或招募员工。


序号: 7
UID: cv26_76dd103842d90505
音频时长: 7.38
当前状态: pending
原始转写: 印度共产主义革命者统一中心是印度的一个共产主义政党。


序号: 8
UID: cv26_93011ddb77e8016e
音频时长: 8.496
当前状态: pending
原始转写: 这也是明朝最后一次派遣琉球册封使团。


序号: 9
UID: cv26_12416afdf8547b3f
音频时长: 5.952
当前状态: pending
原始转写: 现在有试图重建该镇的计划。


序号: 10
UID: cv26_7b77f50a374ed471
音频时长: 5.496
当前状态: pending
原始转写: 杜氏鲾为鲾科鲾属的鱼类。


In [5]:
#状态更新
update_candidate(1,"智秀深信不疑丈夫俊彪是一个标准的好男人。",status="verified")
update_candidate(2,"黄翅蜻属是蜻蜓科下的一个属。",status="verified")
update_candidate(3,"元贞年间再改为横州。",status="verified")
update_candidate(4,"基督教法西斯主义是基督教和法西斯主义的有争议混成词。",status="verified")
update_candidate(5,"它的人员也常因非法闯入这一类罪行被逮捕。",status="verified")
update_candidate(6,"它同时还能够遏制企业在外州建厂或招募员工。",status="verified")
update_candidate(7,"印度共产主义革命者统一中心是印度的一个共产主义政党。",status="verified")
update_candidate(8,"这也是明朝最后一次派遣琉球册封使团。",status="verified")
update_candidate(9,"现在有试图重建该镇的计划。",status="verified")
update_candidate(10,"杜氏鲾为鲾科鲾属的鱼类。",status="verified")

第1条已更新为：verified
第2条已更新为：verified
第3条已更新为：verified
第4条已更新为：verified
第5条已更新为：verified
第6条已更新为：verified
第7条已更新为：verified
第8条已更新为：verified
第9条已更新为：verified
第10条已更新为：verified


In [6]:
#验收代码，检查状态以及标注一致性
from pathlib import Path
import pandas as pd

root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent

candidate_path = root / "metadata" / "test_candidates.csv"

df = pd.read_csv(
    candidate_path,
    dtype=str,
    keep_default_na=False
)

#只检查W5D4的10条
today = df.loc[
    df["review_batch"] == "W5D4"
].copy()

today["selection_rank"] = today["selection_rank"].astype(int)
today = today.sort_values("selection_rank")

print("W5D4候选数量：",len(today))

print("\n状态统计:")
print(today["status"].value_counts())

print("\n逐条状态:")
print(
    today[
        ["selection_rank","status","review_minutes"]
    ].to_string(index=False)
)

#检查标注一致性
problems = []

if len(today) != 10:
    problems.append("W5D4候选数量不是10条")

if today["candidate_uid"].duplicated().any():
    problems.append("存在重复candidate_uid")

for _, row in today.iterrows():
    rank = row["selection_rank"]
    status = row["status"]
    raw_text = row["raw_text"].strip()
    uncertain = row["uncertain_span"].strip()
    notes = row["notes"].strip()

    if status == "pending":
        problems.append(f"第{rank}条仍为pending")

    elif status == "verified":
        if not raw_text:
            problems.append(f"第{rank}条verified但raw_text为空")
        if uncertain:
            problems.append(f"第{rank}条verified但存在不确定片段")

    elif status == "needs_review":
        if not uncertain and not notes:
            problems.append(f"第{rank}条needs_review但没有记录原因")

    else:
        problems.append(f"第{rank}条存在未知状态: {status}")

print("\n验收结果:")
if problems:
    for problem in problems:
        print("需检查:", problem)
else:
    print("标注状态及字段一致性检查通过")

verified_count = (today["status"] == "verified").sum()
print(f"\nverified数量: {verified_count}/10")

W5D4候选数量： 10

状态统计:
status
verified    10
Name: count, dtype: int64

逐条状态:
 selection_rank   status review_minutes
              1 verified               
              2 verified               
              3 verified               
              4 verified               
              5 verified               
              6 verified               
              7 verified               
              8 verified               
              9 verified               
             10 verified               

验收结果:
标注状态及字段一致性检查通过

verified数量: 10/10
